# DS4 Information Theory (InT) -- Stage 4 Step 1

**Notebook:** `52_ds4_information_theory.ipynb`  
**Model family:** Information Theory (InT)  
**Dataset:** DS4 -- Large Naturalistic OneSec Users (N=1,039)  
**Prerequisite:** DS4 StoP (53) -- complete, output loaded directly.  
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_int_blueprint.md`

---

## Goal

Compute 5 entropy quantities plus one DS4-only mutual-information addition:
`H_out_intervention`, `H_out_all_classes`, `H_app`, `H_hour`, `H_emp`, `H_SDE` (exact OU stationary entropy), and `MI(intervention_type; outcome)`.

## Not the goal

- No broader MI/KL program -- deferred, same reasoning as DS5's own FLAG 4 deferral (no textual basis in the shared theory paper).
- No weekly-scale H_SDE -- StoP's weekly sensitivity file does not export a weekly sigma_hat_OU; this notebook states the daily/weekly timescale caveat explicitly instead of re-deriving StoP content.

## Core claim

> H_SDE here is the EXACT differential entropy of the OU stationary distribution N(mu,sigma^2/2theta) -- not an approximation like DS5's H_SDE, which had no true stationary distribution to derive from. H_out splits into an intervention-class-only view (matches StoP's own population) and an all-classes-pooled view (explicitly flagged as mixing logging paths DS4 itself warns against) -- DS5 never needed this split since it has one event class.

## Output

```
outputs/model_outputs/ds4_information_theory/
  participant_entropy_metrics.csv
  int_stop_bridge_summary.csv
  intervention_type_outcome_mi.csv
  model_run_summary.md
  figures/
```


## FLAG decisions

| Flag | Decision | Justification |
|---|---|---|
| FLAG 1 | Report both H_out_intervention (primary) and H_out_all_classes (exploratory) | DS4's 3-class architecture has no DS5 analog |
| FLAG 2 | K=10 bins for H_emp | Method-comparable with DS5; X_t construction is DS4's own |
| FLAG 3 | H_SDE uses daily-scale OU fit only | Weekly sensitivity file has no weekly sigma_hat_OU; caveat stated explicitly instead |
| FLAG 4 | MI/KL apparatus deferred, except MI(intervention_type;outcome) | Matches DS5's FLAG 4 deferral; the one addition is new ground DS5 structurally cannot have |


## DS4 structural cautions

- `intervention_event_class == 'intervention_event'` rows have **only** opened/dismissed outcomes (verified: 603,079 opened + 306,869 dismissed, zero closed, zero dismissed_other). `app_close_event` is 100% closed. `dismissed_other` (812 rows) sits entirely inside `no_intervention`.
- No experimental condition -- same as StoP, all results are within/cross-participant heterogeneity.
- `interventionType` is only meaningfully populated within the `intervention_event` class -- MI(type;outcome) is computed on that subset only.


## Theory

**Shannon entropy:** $H = -\sum_i p_i \log_2 p_i$

**OU stationary differential entropy:** for $X \sim \mathcal{N}(\mu, \sigma_s^2)$ with $\sigma_s^2 = \sigma^2/2\theta$ (the exact OU stationary variance),

$$H_{SDE} = \tfrac{1}{2}\log_2(2\pi e\, \sigma_s^2) = \tfrac{1}{2}\log_2\!\left(2\pi e \,\frac{\sigma^2}{2\theta}\right)$$

This is exact for DS4 (the OU stationary distribution really is Gaussian). DS5's version, `0.5*log2(2*pi*e*sig_hat^2)`, treated `sig_hat` as a free-diffusion increment std -- an approximation, since DS5's constant-drift SDE has no true stationary distribution.

**Mutual information:** $I(X;Y) = \sum_{x,y} p(x,y)\log_2\frac{p(x,y)}{p(x)p(y)} \geq 0$


## Theory-to-data mapping

| Quantity | DS4 source | Scope |
|---|---|---|
| H_out_intervention | `event_outcome_simple` | `intervention_event` class only, 2 categories |
| H_out_all_classes | `event_outcome_simple` | all classes pooled, 4 categories, exploratory only |
| H_app | Stage 1 `app_entropy` | all events, unfiltered |
| H_hour | Stage 1 `hour_entropy` | all events, unfiltered |
| H_emp | binned daily X_t (K=10) | same population as StoP |
| H_SDE | StoP's sigma_hat_OU/theta_hat_OU | daily scale only |
| MI(type;outcome) | `interventionType` x `event_outcome_simple` | `intervention_event` class only |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BEH_PATH = Path("outputs/cleaned/ds4_behavioral_clean.csv")
FEAT_PATH = Path("outputs/features/ds4/ds4_behavior_features.csv")
STOP_PATH = Path("outputs/model_outputs/ds4_stochastic_processes/participant_parameters.csv")
OUT_DIR = Path("outputs/model_outputs/ds4_information_theory")
FIG_DIR = OUT_DIR / "figures"

K_BINS = 10

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


def shannon_entropy(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total <= 0:
        return np.nan
    p = counts[counts > 0] / total
    return float(-(p * np.log2(p)).sum())


In [ ]:
beh = pd.read_csv(BEH_PATH, parse_dates=["timestamp_utc"], low_memory=False)
stage1_feats = pd.read_csv(FEAT_PATH)
stop_params = pd.read_csv(STOP_PATH)

print(f"beh: {len(beh):,} rows, {beh['participant_id_canonical'].nunique()} participants")
print(f"stage1_feats: {len(stage1_feats)} rows")
print(f"stop_params: {len(stop_params)} rows, {int(stop_params['fit_valid'].sum())} valid fits")


In [ ]:
ie = beh[beh["intervention_event_class"] == "intervention_event"]

ie_counts = ie.groupby(["participant_id_canonical", "event_outcome_simple"]).size().unstack(fill_value=0)
h_out_intervention = ie_counts.apply(lambda row: shannon_entropy(row.values), axis=1)
h_out_intervention = h_out_intervention.rename("H_out_intervention").reset_index()

all_counts = beh.groupby(["participant_id_canonical", "event_outcome_simple"]).size().unstack(fill_value=0)
h_out_all = all_counts.apply(lambda row: shannon_entropy(row.values), axis=1)
h_out_all = h_out_all.rename("H_out_all_classes").reset_index()

print(f"H_out_intervention: {len(h_out_intervention)} participants")
print(f"H_out_all_classes: {len(h_out_all)} participants")


In [ ]:
h_app_hour = stage1_feats[["participant_id_canonical", "app_entropy", "hour_entropy"]].rename(
    columns={"app_entropy": "H_app", "hour_entropy": "H_hour"}
)
print(f"H_app/H_hour loaded for {len(h_app_hour)} participants")


In [ ]:
df = beh[beh["intervention_event_class"] == "intervention_event"].copy()
df = df[df["event_outcome_simple"].isin(["opened", "dismissed"])].copy()
df["opened_flag"] = (df["event_outcome_simple"] == "opened").astype(int)
df["event_date"] = df["timestamp_utc"].dt.date

daily = df.groupby(["participant_id_canonical", "event_date"]).agg(
    n_opened=("opened_flag", "sum"),
    n_events=("opened_flag", "count"),
).reset_index()
daily["X_t"] = daily["n_opened"] / daily["n_events"]

bin_edges = np.linspace(0, 1, K_BINS + 1)

def h_emp_for_group(g):
    counts, _ = np.histogram(g["X_t"].values, bins=bin_edges)
    return shannon_entropy(counts)

h_emp = daily.groupby("participant_id_canonical").apply(h_emp_for_group)
h_emp = h_emp.rename("H_emp").reset_index()
print(f"H_emp computed for {len(h_emp)} participants")


In [ ]:
theta = stop_params["theta_hat_OU"]
sigma = stop_params["sigma_hat_OU"]
stationary_var = sigma ** 2 / (2 * theta)
stop_params["H_SDE"] = 0.5 * np.log2(2 * np.pi * np.e * stationary_var)

n_valid = int(stop_params["fit_valid"].sum())
n_finite = int(np.isfinite(stop_params.loc[stop_params["fit_valid"], "H_SDE"]).sum())
print(f"H_SDE finite for {n_finite} of {n_valid} valid StoP fits")


In [ ]:
merged_check = stop_params.merge(h_out_intervention, on="participant_id_canonical", how="left")
valid_check = merged_check[merged_check["fit_valid"]]
corr_hout_mu = valid_check["H_out_intervention"].corr(valid_check["mu_hat_OU"])
print(f"corr(H_out_intervention, mu_hat_OU) among valid StoP fits: {corr_hout_mu:.3f} "
      f"(n={valid_check['H_out_intervention'].notna().sum()})")


In [ ]:
mi_df = ie.dropna(subset=["interventionType"]).copy()
contingency = pd.crosstab(mi_df["interventionType"], mi_df["event_outcome_simple"])
print(contingency)

def mutual_information(contingency_table):
    total = contingency_table.values.sum()
    p_xy = contingency_table.values / total
    p_x = p_xy.sum(axis=1, keepdims=True)
    p_y = p_xy.sum(axis=0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(p_xy > 0, p_xy / (p_x * p_y), 1.0)
        terms = np.where(p_xy > 0, p_xy * np.log2(ratio), 0.0)
    return float(terms.sum())

mi_type_outcome = mutual_information(contingency)
print(f"\nMI(intervention_type; outcome) = {mi_type_outcome:.4f} bits")
assert mi_type_outcome >= -1e-9, "MI must be non-negative"

type_rates = contingency.div(contingency.sum(axis=1), axis=0).reset_index()
type_rates = type_rates.rename(columns={"interventionType": "intervention_type"})
type_rates["n_events"] = contingency.sum(axis=1).values
print(type_rates)


In [ ]:
entropy_metrics = stop_params[[
    "participant_id_canonical", "mu_hat_OU", "theta_hat_OU", "sigma_hat_OU",
    "tau_hat", "fit_valid", "H_SDE"
]].copy()
entropy_metrics = entropy_metrics.merge(h_out_intervention, on="participant_id_canonical", how="left")
entropy_metrics = entropy_metrics.merge(h_out_all, on="participant_id_canonical", how="left")
entropy_metrics = entropy_metrics.merge(h_app_hour, on="participant_id_canonical", how="left")
entropy_metrics = entropy_metrics.merge(h_emp, on="participant_id_canonical", how="left")

entropy_metrics.to_csv(OUT_DIR / "participant_entropy_metrics.csv", index=False)
print(f"Wrote participant_entropy_metrics.csv ({len(entropy_metrics)} rows)")


In [ ]:
entropy_cols = ["H_out_intervention", "H_out_all_classes", "H_app", "H_hour", "H_emp", "H_SDE"]
dist_summary_int = entropy_metrics[entropy_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print(dist_summary_int)


In [ ]:
valid_entropy = entropy_metrics[entropy_metrics["fit_valid"]]
r_hemp_hsde = valid_entropy["H_emp"].corr(valid_entropy["H_SDE"])
r_hout_mu = valid_entropy["H_out_intervention"].corr(valid_entropy["mu_hat_OU"])

bridge_summary = pd.DataFrame([{
    "r_H_emp_H_SDE": r_hemp_hsde,
    "r_H_out_intervention_mu_hat_OU": r_hout_mu,
    "n_valid": int(valid_entropy["H_SDE"].notna().sum()),
}])
bridge_summary.to_csv(OUT_DIR / "int_stop_bridge_summary.csv", index=False)
print(bridge_summary)

summary_row = pd.DataFrame([{
    "intervention_type": "__ALL__",
    "opened": np.nan, "dismissed": np.nan,
    "n_events": int(contingency.values.sum()),
    "MI_bits": mi_type_outcome,
}])
type_rates_export = type_rates.copy()
type_rates_export["MI_bits"] = np.nan
mi_export = pd.concat([summary_row, type_rates_export], ignore_index=True)
mi_export.to_csv(OUT_DIR / "intervention_type_outcome_mi.csv", index=False)
print(mi_export)


In [ ]:
summary_lines = []
summary_lines.append("# DS4 InT -- Model Run Summary")
summary_lines.append("")
summary_lines.append("**Notebook:** `52_ds4_information_theory.ipynb`  ")
summary_lines.append("**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_int_blueprint.md`  ")
summary_lines.append("**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"))
summary_lines.append("")
summary_lines.append("## Entropy distribution summary")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(dist_summary_int.to_string())
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## Bridge / redundancy checks")
summary_lines.append("")
summary_lines.append(f"- r(H_emp, H_SDE) = {r_hemp_hsde:.3f} "
                     f"(n={int(valid_entropy['H_SDE'].notna().sum())})")
summary_lines.append(f"- r(H_out_intervention, mu_hat_OU) = {r_hout_mu:.3f}")
summary_lines.append("")
summary_lines.append("## MI(intervention_type; outcome)")
summary_lines.append("")
summary_lines.append(f"MI = {mi_type_outcome:.4f} bits (n_events={int(contingency.values.sum())})")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(type_rates.to_string())
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## Timescale caveat (carried forward from StoP)")
summary_lines.append("")
summary_lines.append("H_SDE uses daily-scale sigma_hat_OU/theta_hat_OU only. StoP's own "
                     "weekly sensitivity check found a meaningfully different timescale "
                     "(median tau ~0.63 days daily vs ~0.91 weeks weekly). H_SDE here "
                     "describes daily-timescale uncertainty specifically, not a "
                     "timescale-independent property.")

with open(OUT_DIR / "model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote model_run_summary.md")


In [ ]:
fig_data = entropy_metrics.merge(
    stage1_feats[["participant_id_canonical", "intervention_event_share"]],
    on="participant_id_canonical", how="left",
)
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(fig_data["H_out_intervention"], fig_data["H_out_all_classes"],
                  c=fig_data["intervention_event_share"], cmap="plasma", s=14, alpha=0.7)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("intervention_event_share")
ax.plot([0, 2], [0, 2], color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("H_out_intervention (bits)")
ax.set_ylabel("H_out_all_classes (bits)")
ax.set_title("DS4 InT: outcome entropy, intervention-only vs all-classes-pooled")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ds4_hout_comparison.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_data = valid_entropy.dropna(subset=["H_emp", "H_SDE"])
ax.scatter(plot_data["H_SDE"], plot_data["H_emp"], s=14, alpha=0.7, color="#4C72B0")
ax.set_xlabel("H_SDE (bits, OU stationary entropy)")
ax.set_ylabel("H_emp (bits, empirical binned entropy)")
ax.set_title(f"DS4 InT-StoP bridge: H_emp vs H_SDE (r={r_hemp_hsde:.3f}, n={len(plot_data)})")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_ds4_hemp_hsde_bridge.png", dpi=150)
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(entropy_metrics["H_app"].dropna(), bins=40, color="#55A868", edgecolor="white")
axes[0].set_xlabel("H_app (bits)")
axes[0].set_ylabel("count")
axes[0].set_title("App entropy")

axes[1].hist(entropy_metrics["H_hour"].dropna(), bins=40, color="#C44E52", edgecolor="white")
axes[1].set_xlabel("H_hour (bits)")
axes[1].set_title("Hour-of-day entropy")

fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_ds4_happ_hhour_dist.png", dpi=150)
plt.close(fig)


In [ ]:
overall_opened_rate = contingency["opened"].sum() / contingency.values.sum()

fig, ax = plt.subplots(figsize=(8, 5))
type_rates_sorted = type_rates.sort_values("opened")
ax.barh(type_rates_sorted["intervention_type"], type_rates_sorted["opened"], color="#8172B2")
ax.axvline(overall_opened_rate, color="black", linestyle="--", linewidth=1,
            label=f"overall opened rate ({overall_opened_rate:.2f})")
ax.set_xlabel("opened rate")
ax.set_title(f"Opened rate by intervention type (MI={mi_type_outcome:.4f} bits)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_ds4_mi_by_type.png", dpi=150)
plt.close(fig)


## Interpretation and limits

**Read the cells above for live numbers** -- nothing here is hardcoded.

- `H_SDE` is daily-timescale specific (see the model_run_summary.md timescale caveat). Do not present it as a timescale-independent bridge quantity.
- `H_out_intervention` and `mu_hat_OU` share real but partial variance (cell 12's live correlation) -- report both, neither is redundant with the other.
- `H_out_all_classes` mixes structurally different event-logging paths on purpose, to show what the intervention-only view leaves out -- exploratory, not primary.
- `MI(intervention_type; outcome)` is correlational. Intervention type in DS4 is not randomly assigned -- do not read this as a causal claim about which intervention type is more effective.

### DS4 weakness flags

- FLAG I: H_out_all_classes descriptive only.
- FLAG II: H_SDE daily-timescale only.
- FLAG III: H_out_intervention / mu_hat_OU partial redundancy.
- FLAG IV: MI(type;outcome) correlational, not causal.
- FLAG V: all entropy measures inherit X_t's behavioral-proxy limitation.


## Forward connections

| Connection | Mechanism | Target |
|---|---|---|
| DS4 InT -> DS4 DyS | H_emp as model-selection criterion between deterministic/stochastic regimes | `55_ds4_dynamical_systems.ipynb` |
| DS4 InT -> DS4 StM | H_SDE formula identical to `ds4_stm_blueprint.md`'s own H_StM -- StM reuses it for its convergence check | `51_ds4_statistical_mechanics.ipynb` |
| DS4 InT -> Paper | Exact OU stationary entropy formula and the daily/weekly timescale caveat are not yet in the conceptual paper |

**Next step:** write the DS4 DyS (55) blueprint, informed by this notebook's real H_emp output.
